# MOSS: OpenVINO Intel GPU experiment

Compare the **same original MOSS weights** using PyTorch **BF16 XPU / math SDPA** and OpenVINO **FP16 GPU**. This is a runtime and precision comparison, not a BF16-for-BF16 comparison. Neither trial uses INT8/FP8 quantization. OpenVINO uses a custom exported Whisper encoder/adaptor and Qwen3 decoder with **stateful KV cache** and greedy generation. Finished sequences stop independently; cache resets between batches.

Every model operation runs in a disposable worker with cleanup on success, failure, and interruption. Model conversion, graph compilation, preprocessing, encoder, prefill, token decode, text decoding, cleanup, and export are timed separately. Instrumentation affects latency; a single run is exploratory, not a controlled benchmark.

Install optional dependencies from the project root before running:
```bash
uv pip install --python .venv -r requirements-openvino.txt --torch-backend=xpu
```
Select the project `.venv` kernel. Direct conversion avoids installing Optimum and changing the existing Transformers version. The first export needs the source model and approximately 2 GiB of cached IR assets, plus temporary RAM.


In [1]:
import sys
import json
import tempfile
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
from dataclasses import asdict
from difflib import SequenceMatcher
from html import escape
from IPython.display import HTML, display
import openvino as ov
import torch
import transformers
import soundfile as sf

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "speech_to_text").is_dir():
    raise RuntimeError("Run this notebook from the project root.")
if Path(sys.prefix).resolve() != (PROJECT_ROOT / ".venv").resolve():
    raise RuntimeError("Select the project-local .venv kernel.")
from speech_to_text.chunked import transcribe_chunked
from speech_to_text.outputs import export_results
from speech_to_text.openvino_export import export_isolated
from speech_to_text.profiling import StageProfiler

MODEL_ID = "OpenMOSS-Team/MOSS-Transcribe-Diarize"
IR_DIR = PROJECT_ROOT / ".tools" / "moss-openvino-fp16"
AUDIO_DIR = PROJECT_ROOT / "audio" / "active"
EXTENSIONS = {".wav", ".mp3", ".flac", ".m4a", ".ogg", ".aac", ".opus", ".wma"}
audio_files = sorted(p for p in AUDIO_DIR.rglob("*") if p.is_file()
                     and p.suffix.lower() in EXTENSIONS and not p.is_symlink()
                     and p.resolve().is_relative_to(AUDIO_DIR.resolve()))
CHUNK_SECONDS = 30
BATCH_SIZE = 1
CPU_THREADS = 1
MAX_NEW_TOKENS = 1024
LANGUAGES = ["Thai", "English"]
REPEATS = 1
OV_DEVICE = "GPU"  # Fail explicitly if GPU is unavailable; no automatic CPU fallback.
FORCE_EXPORT = False
WORKER_TIMEOUT_SECONDS = 1200
OUTPUT_BASE = PROJECT_ROOT / "outputs"
OUTPUT_BASE.mkdir(exist_ok=True)
report_dir = Path(tempfile.mkdtemp(prefix="openvino-iGPU-", dir=OUTPUT_BASE))
print("Versions:", {"openvino": ov.__version__, "torch": torch.__version__, "transformers": transformers.__version__})
print("Inputs:", [str(p) for p in audio_files])
print("Reports:", report_dir)


Versions: {'openvino': '2026.4.1-22982-e213a147257-releases/2026/4', 'torch': '2.15.0+xpu', 'transformers': '5.22.1'}
Inputs: ['/mnt/data/apps/MOSS-ASR-XPU/audio/active/macos_mic.m4a']
Reports: /mnt/data/apps/MOSS-ASR-XPU/outputs/openvino-iGPU-hkykj4dw


## 1. Convert once, then reuse the IR cache

Conversion loads the original model on CPU in FP32, saves FP16 constants and embeddings, and unloads it. This one-time cost is excluded from each inference trial. The manifest records the source snapshot, dependency versions, precision, and original export time. A cache is reused only when all required files, model ID, format, and conversion versions match. Set `FORCE_EXPORT=True` after changing the exporter.

The source processor still performs the model-required 16 kHz mono preprocessing. Native chunk files and speaker WAV exports retain the source sample rate, channels, and PCM/float subtype. Compressed sources are decoded to WAV; their compressed codec/bitrate is not preserved.


In [2]:
required = ["manifest.json", "encoder.xml", "encoder.bin", "decoder.xml", "decoder.bin", "embeddings.npy", "processor_config.json"]
manifest = json.loads((IR_DIR / "manifest.json").read_text()) if (IR_DIR / "manifest.json").is_file() else {}
cache_valid = (all((IR_DIR / name).is_file() for name in required)
               and manifest.get("format_version") == 1 and manifest.get("model_id") == MODEL_ID
               and manifest.get("openvino_version") == ov.__version__
               and manifest.get("torch_version") == torch.__version__
               and manifest.get("transformers_version") == transformers.__version__)
conversion_started = perf_counter()
reused_export = cache_valid and not FORCE_EXPORT
if not reused_export:
    manifest = export_isolated(MODEL_ID, IR_DIR)
conversion = {"reused_export": reused_export, "export_or_cache_check_seconds": perf_counter() - conversion_started,
              "original_export_seconds": manifest["export_seconds"], "manifest": manifest}
(report_dir / "conversion.json").write_text(json.dumps(conversion, indent=2))
print(json.dumps(conversion, indent=2))


{
  "reused_export": true,
  "export_or_cache_check_seconds": 7.007299791439436e-05,
  "original_export_seconds": 40.422821028001636,
  "manifest": {
    "format_version": 1,
    "model_id": "OpenMOSS-Team/MOSS-Transcribe-Diarize",
    "source_snapshot": "10304d926740cf5a5cee05a268086e3265197910",
    "openvino_version": "2026.4.1-22982-e213a147257-releases/2026/4",
    "torch_version": "2.15.0+xpu",
    "transformers_version": "5.22.1",
    "weight_storage": "FP16",
    "inference_precision_hint": "FP16",
    "kv_cache": "OpenVINO stateful; reset independently for every batch",
    "num_layers": 28,
    "kv_heads": 8,
    "head_dim": 128,
    "audio_token_id": 151671,
    "eos_token_id": 151645,
    "pad_token_id": 151643,
    "export_seconds": 40.422821028001636
  }
}


## 2. Run both backends on the same complete recordings

Each trial loads/compiles once, processes every native audio window, exports speaker WAV/text/JSON, checks cleanup, and exits. There is no separate warmup: the first batch includes first-use overhead. `processing_seconds` includes input processing through parsed text, excluding load/compile, window writing, cleanup, and exports; `workflow_seconds` includes all those costs.

OpenVINO workers use the **system OpenCL runtime**. The optional project Level Zero compiler libraries used by PyTorch XPU are removed only from the OpenVINO worker's library search path: using them with OpenVINO caused GPU kernel compilation failures on this machine. The encoder and decoder must both report GPU execution.

The two runtimes have separate allocators. PyTorch's zero allocation values do **not** measure OpenVINO GPU memory. OpenVINO records its own GPU memory statistics before compile, before unload, and after unload. A driver context/pool can survive unloading until worker exit.

The default batch size is 1 to fit the current shared system-memory budget. An initial 30-second batch-2 baseline hit the worker RAM reserve; an 8-second OpenVINO batch-2 smoke test passed. Increase batch size only when sufficient RAM is available.


In [3]:
if not audio_files or REPEATS < 1:
    raise ValueError("Provide active audio inputs and REPEATS >= 1.")
trials = []
try:
    for audio_path in audio_files:
        for repeat in range(1, REPEATS + 1):
            for backend in ("pytorch", "openvino"):
                run_dir = Path(tempfile.mkdtemp(prefix=f"{backend}-{repeat}-", dir=report_dir))
                parent = StageProfiler(run_dir / "workflow")
                trial = {"source": str(audio_path), "backend": backend, "repeat": repeat,
                         "run_dir": str(run_dir), "created_at": datetime.now(timezone.utc).isoformat(),
                         "status": "running"}
                trials.append(trial)
                started = perf_counter()
                try:
                    options = ({"ov_model_dir": str(IR_DIR), "ov_device": OV_DEVICE}
                               if backend == "openvino" else {})
                    with parent.stage("isolated_worker"):
                        raw, segments, metrics = transcribe_chunked(
                            audio_path, MODEL_ID, "xpu:0", MAX_NEW_TOKENS, LANGUAGES,
                            chunk_seconds=CHUNK_SECONDS, batch_size=BATCH_SIZE, cpu_threads=CPU_THREADS,
                            quantization="none", use_cache=True, backend=backend,
                            profile_output_dir=run_dir / "worker", timeout_seconds=WORKER_TIMEOUT_SECONDS,
                            **options)
                    if backend == "openvino":
                        devices = metrics["encoder_execution_devices"] + metrics["decoder_execution_devices"]
                        if OV_DEVICE.startswith("GPU") and not all(d.startswith("GPU") for d in devices):
                            raise RuntimeError(f"Unexpected execution devices: {devices}")
                    with parent.stage("speaker_and_transcript_export"):
                        export_results(audio_path, run_dir, raw, segments)
                    with parent.stage("native_export_validation"):
                        source = metrics["source_audio"]
                        exported = sorted(run_dir.glob("S*.wav"))
                        if not exported:
                            raise RuntimeError("No speaker WAV outputs were produced.")
                        for wav in exported:
                            info = sf.info(wav)
                            if (info.samplerate, info.channels, info.subtype) != (
                                    source["sample_rate"], source["channels"], source["subtype"]):
                                raise RuntimeError(f"Native audio properties changed: {wav}")
                    trial.update(status="success", metrics=metrics, segments=[asdict(s) for s in segments],
                                 workflow_seconds=perf_counter() - started)
                    (run_dir / "experiment.json").write_text(json.dumps(trial, indent=2, ensure_ascii=False))
                    print(backend, "processing:", round(metrics["chunk_processing_seconds"], 3),
                          "workflow:", round(trial["workflow_seconds"], 3), "segments:", len(segments))
                except BaseException as error:
                    trial.update(status="failed", error=repr(error), workflow_seconds=perf_counter() - started)
                    raise
                finally:
                    parent.save()
                    (report_dir / "trials.json").write_text(json.dumps(trials, indent=2, ensure_ascii=False))
finally:
    print("Saved results and any partial profiles:", report_dir)


CPU threads: 1; inter-op threads: 1
Loading MOSS once on xpu:0; quantization=none


Loading weights: 100%|██████████| 683/683 [00:00<00:00, 4037.09it/s]
[transformers] `MossTranscribeDiarizeProcessor` defines `feature_extractor_class = 'AutoFeatureExtractor'`, which is deprecated. Register the correct mapping in `AutoFeatureExtractor` instead.


Batch 1: chunks 1-1/1
Batch generated tokens: [163]
Batch output: 35.255s; audio 24.000s; RTF 1.469
Batch PyTorch allocator: 1817041920 bytes (baseline: 1817041920)
pytorch processing: 35.255 workflow: 45.114 segments: 6
CPU threads: 1; inter-op threads: 1


[transformers] `MossTranscribeDiarizeProcessor` defines `feature_extractor_class = 'AutoFeatureExtractor'`, which is deprecated. Register the correct mapping in `AutoFeatureExtractor` instead.


OpenVINO execution: {'backend': 'openvino', 'ov_device': 'GPU', 'torch_allocator_scope': 'CPU preprocessing only; not OpenVINO GPU memory', 'openvino_batches': [], 'ov_version': '2026.4.1-22982-e213a147257-releases/2026/4', 'ov_device_name': 'Intel(R) Graphics (iGPU)', 'ov_memory_before_compile': {'cl_mem': 0, 'unknown': 0, 'usm_device': 0, 'usm_host': 0, 'usm_shared': 0}, 'encoder_execution_devices': ['GPU.0'], 'decoder_execution_devices': ['GPU.0']}
Batch 1: chunks 1-1/1
OpenVINO generated 1/1024 steps
OpenVINO generated 128/1024 steps
Batch output: 8.167s; audio 24.000s; RTF 0.340
Batch PyTorch allocator: 0 bytes (baseline: 0)
openvino processing: 8.167 workflow: 17.458 segments: 6
Saved results and any partial profiles: /mnt/data/apps/MOSS-ASR-XPU/outputs/openvino-iGPU-hkykj4dw


## 3. Compare timings and inspect every measured stage

PyTorch prefill includes the encoder/adaptor; OpenVINO prefill times the language decoder after a separately measured encoder and embedding injection. Decode timings include completed forward work and, for OpenVINO, argmax; PyTorch hooks measure forward calls. These scopes differ slightly. Compare whole processing/workflow time first. Token counts may differ when FP16/BF16 rounding changes greedy choices.

The stage table below lists inclusive and exclusive wall time for **every stage**. Nested inclusive times must not be added together. OpenVINO synchronous infer calls already wait for device completion; PyTorch hooks explicitly synchronize XPU. CSV/JSON reports retain per-call events and per-batch counts.


In [4]:
comparison = []
for trial in trials:
    if trial["status"] != "success":
        continue
    metrics = trial["metrics"]
    profile = json.loads((Path(trial["run_dir"]) / "worker" / "profile.json").read_text())
    totals = {row["stage"]: row["inclusive_seconds"] for row in profile["summary"]}
    batches = metrics.get("openvino_batches", profile["batches"])
    tokens = sum(sum(b.get("generated_tokens", [])) for b in batches)
    comparison.append({"source": Path(trial["source"]).name, "backend": trial["backend"],
                       "repeat": trial["repeat"], "precision": metrics["dtype"],
                       "workflow_seconds": trial["workflow_seconds"],
                       "worker_seconds": metrics["isolated_worker_wall_seconds"],
                       "processing_seconds": metrics["chunk_processing_seconds"],
                       "processing_rtf": metrics["chunk_processing_rtf"],
                       "compile_seconds": totals.get("openvino_encoder_compile", 0) + totals.get("openvino_decoder_compile", 0),
                       "encoder_seconds": totals.get("audio_encoder", totals.get("openvino_audio_encoder", 0)),
                       "prefill_seconds": totals.get("prefill_forward", totals.get("openvino_prefill", 0)),
                       "decode_seconds": totals.get("decode_forward", totals.get("openvino_decode", 0)),
                       "tokens": tokens, "segments": len(trial["segments"]), "run_dir": trial["run_dir"]})
StageProfiler.write_csv(report_dir / "comparison.csv", comparison)
(report_dir / "comparison.json").write_text(json.dumps(comparison, indent=2))
columns = [key for key in comparison[0] if key != "run_dir"]
rows = "".join("<tr>" + "".join(f"<td>{escape(str(round(row[k], 3) if isinstance(row[k], float) else row[k]))}</td>" for k in columns) + "</tr>" for row in comparison)
display(HTML("<table><tr>" + "".join(f"<th>{escape(k)}</th>" for k in columns) + "</tr>" + rows + "</table>"))
for trial in trials:
    path = Path(trial["run_dir"]) / "worker" / "stage_summary.csv"
    print("All stages:", trial["backend"], path)
    print(path.read_text())
    metrics = trial["metrics"]
    memory = ({k: v for k, v in metrics.items() if k.startswith("ov_memory_")}
              if trial["backend"] == "openvino" else
              {k: metrics[k] for k in ("peak_allocated_bytes", "allocated_bytes_after", "reserved_bytes_after")})
    print("Allocator-specific memory:", json.dumps(memory, indent=2))


source,backend,repeat,precision,workflow_seconds,worker_seconds,processing_seconds,processing_rtf,compile_seconds,encoder_seconds,prefill_seconds,decode_seconds,tokens,segments
macos_mic.m4a,pytorch,1,bfloat16,45.114,44.897,35.255,1.469,0,2.051,4.164,29.781,163,6
macos_mic.m4a,openvino,1,float16,17.458,17.378,8.167,0.34,3.554,1.181,0.204,6.462,163,6


All stages: pytorch /mnt/data/apps/MOSS-ASR-XPU/outputs/openvino-iGPU-hkykj4dw/pytorch-1-_z_ja21v/worker/stage_summary.csv
stage,calls,inclusive_seconds,exclusive_seconds
worker_imports,1,0.9686219049981446,0.9686219049981446
worker_runtime_initialization,1,0.043502494001586456,0.043502494001586456
model_weights_load,1,1.087621463000687,1.087621463000687
model_to_device,1,1.6502075890020933,1.6502075890020933
processor_load,1,2.2762593970001035,2.2762593970001035
audio_source_open_and_decode,1,0.3973191580007551,0.3973191580007551
chunk_write,1,0.014359200999024324,0.014359200999024324
prompt_and_chat_template,1,0.023399713001708733,0.023399713001708733
audio_read_resample_downmix,1,0.19063372399978107,0.19063372399978107
processor_features_and_tokenization,1,0.008979285001259996,0.008979285001259996
left_padding,1,0.0001416749983036425,0.0001416749983036425
host_to_device,1,0.0025998980017902795,0.0025998980017902795
generation,1,35.027524034001544,1.082296935004706
prefill_forward,1,

## 4. Compare transcripts before choosing a runtime

Text agreement is **not WER, DER, or accuracy**: no reference transcript/speaker labels are provided. Inspect both transcripts, timestamps, and speaker boundaries. Speakers are independent per chunk in both backends. A different token count or faster but incomplete transcript must not be treated as an equivalent result. The generation loop raises an error at the token limit rather than silently accepting truncated output.


In [5]:
agreements = []
for audio_path in audio_files:
    for repeat in range(1, REPEATS + 1):
        pair = {t["backend"]: t for t in trials if t["source"] == str(audio_path) and t["repeat"] == repeat and t["status"] == "success"}
        if len(pair) != 2:
            continue
        texts = {key: " ".join(s["text"] for s in trial["segments"]) for key, trial in pair.items()}
        agreement = {"source": str(audio_path), "repeat": repeat,
                     "exact_text_match": texts["pytorch"] == texts["openvino"],
                     "character_similarity_not_accuracy": SequenceMatcher(None, texts["pytorch"], texts["openvino"], autojunk=False).ratio(),
                     "processing_speed_ratio_pytorch_over_openvino": pair["pytorch"]["metrics"]["chunk_processing_seconds"] / pair["openvino"]["metrics"]["chunk_processing_seconds"]}
        agreements.append(agreement)
        print(json.dumps(agreement, indent=2))
        display(HTML("<table><tr><th>PyTorch BF16</th><th>OpenVINO FP16</th></tr><tr>" +
                     "".join(f"<td style='vertical-align:top;white-space:pre-wrap'>{escape((Path(pair[k]['run_dir']) / 'transcript.txt').read_text())}</td>" for k in ("pytorch", "openvino")) + "</tr></table>"))
(report_dir / "agreement.json").write_text(json.dumps(agreements, indent=2))
print("Comparison report:", report_dir / "comparison.csv")
print("Conversion report:", report_dir / "conversion.json")
print("Per-call timings: each trial's worker/stages.csv and workflow/stages.csv")


{
  "source": "/mnt/data/apps/MOSS-ASR-XPU/audio/active/macos_mic.m4a",
  "repeat": 1,
  "exact_text_match": true,
  "character_similarity_not_accuracy": 1.0,
  "processing_speed_ratio_pytorch_over_openvino": 4.316799466349884
}


PyTorch BF16,OpenVINO FP16
"[0.12-1.82] S01: Subway สวัสดีค่ะ [2.38-4.94] S01: รับอะไรเพิ่มไหมคะ Hello, how are you today? [5.78-8.52] S01: Um, what would you like? [9.88-13.22] S01: Good morning, Subway, good morning, what would you like? [13.88-16.84] S01: Uh, cucumber or uh honey bread? [17.73-22.94] S01: Uh, รับอะไรดีคะ Subway สวัสดีค่ะ","[0.12-1.82] S01: Subway สวัสดีค่ะ [2.38-4.94] S01: รับอะไรเพิ่มไหมคะ Hello, how are you today? [5.78-8.52] S01: Um, what would you like? [9.88-13.22] S01: Good morning, Subway, good morning, what would you like? [13.88-16.84] S01: Uh, cucumber or uh honey bread? [17.73-22.94] S01: Uh, รับอะไรดีคะ Subway สวัสดีค่ะ"


Comparison report: /mnt/data/apps/MOSS-ASR-XPU/outputs/openvino-iGPU-hkykj4dw/comparison.csv
Conversion report: /mnt/data/apps/MOSS-ASR-XPU/outputs/openvino-iGPU-hkykj4dw/conversion.json
Per-call timings: each trial's worker/stages.csv and workflow/stages.csv
